<a href="https://colab.research.google.com/github/alxmzr/Colab/blob/main/Automated_Algo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Automated Algorithm v 1.0




Credits to https://gist.github.com/yhilpisch/3c538e90eece733b72bdb46404e8ce2b from which this was adapted

Importing data

In [ ]:
import datetime as dt
import datetime
import numpy as np
import pandas as pd
import math
import time
import cufflinks as cf  # Cufflinks
cf.set_config_file(offline=True)  # set the plotting mode to offline

import plotly.graph_objs as go

In [ ]:
!pip install fxcmpy
import fxcmpy

  Using cached fxcmpy-0.0.0.1.tar.gz (966 bytes)
  Preparing metadata (setup.py) ... done
  Created wheel for fxcmpy: filename=fxcmpy-0.0.0.1-py3-none-any.whl size=1281 sha256=01e4a15989b00cf8fb60b6967e69e1b8a9b5454de683f4148acd817d2ecc10a6
  Stored in directory: /root/.cache/pip/wheels/30/97/76/dc4aa322bc72f98d9a5876f4c33a8997e4408b6546cdb882cd
Successfully built fxcmpy


In [ ]:
#Need to add fxcm.cfg file to same level directory as FXCM same format
#[FXCM]
#log_level = error
#log_file = PATH_TO_AND_NAME_OF_LOG_FILE
#access_token = ""
api = fxcmpy.fxcmpy(config_file='fxcm.cfg')

AttributeError: module 'fxcmpy' has no attribute 'fxcmpy'

In [ ]:
# More documentation http://fxcmpy.tpq.io/
# Calling API to get data
# change start and stop times
# Instrument must be one of ('EUR/USD', 'USD/JPY', 'GBP/USD', 'USD/CHF', '
# EUR/CHF', 'AUD/USD', 'USD/CAD', 'NZD/USD', 'EUR/GBP', 'EUR/JPY', 'GBP/JPY',
#'AUD/JPY', 'USD/CNH', 'FRA40', 'GER30', 'UK100', 'US30', 'USDOLLAR', 'XAU/USD', 'XAG/USD').

candles = api.get_candles('EUR/USD', period='m5',
                         start=dt.datetime(2018, 2, 7),
                          stop=dt.datetime(2018, 2, 22))

NameError: name 'api' is not defined

In [ ]:
#Putting data into a dataframe with a midclose
data = pd.DataFrame(candles[['askclose', 'bidclose']].mean(axis=1),
                    columns=['midclose'])

In [ ]:
data['spread']= candles['askclose'] - candles['bidclose']

In [ ]:
data['spread'].head()

In [ ]:
###OOS
candles_oos = api.get_candles('EUR/USD', period='m5',
                         start=dt.datetime(2018, 6, 23),
                          stop=dt.datetime(2018, 8, 8))
#Putting data into a dataframe with a midclose
data_oos = pd.DataFrame(candles_oos[['askclose', 'bidclose']].mean(axis=1),
                    columns=['midclose'])
data_oos.info()

#data_oos.iplot()



In [ ]:
### OOS
data_oos['returns'] = np.log(data_oos / data_oos.shift(1))
lags = 5

data_oos['spread']= candles_oos['askclose'] - candles_oos['bidclose']

# Applying lag# Normalising Returns
# Applying lag
cols = []
for lag in range(1, lags + 1):
    col = 'lag_%s' % lag
    data_oos[col] = data_oos['returns'].shift(lag)
    cols.append(col)

# Adding features which such as rolling returns
col = 'momentum'
data_oos[col] = data_oos['returns'].rolling(5).mean().shift(1)
cols.append(col)#

cols


In [ ]:
data.info()

In [ ]:
data.tail()

In [ ]:
data.iplot()

# Feature Preperation

In [ ]:
# Normalising Returns
data['returns'] = np.log(data['midclose'] / data.shift(1)['midclose'])

In [ ]:
# Enter the amount of extra data
lags = 5


In [ ]:
# Applying lag
cols = []
for lag in range(1, lags + 1):
    col = 'lag_%s' % lag
    data[col] = data['returns'].shift(lag)
    cols.append(col)

In [ ]:
# Adding features which such as rolling returns
col = 'momentum'
data[col] = data['returns'].rolling(5).mean().shift(1)
cols.append(col)

In [ ]:
cols

# Transformation to binning

In [ ]:
from pylab import plt
plt.style.use('seaborn')
%matplotlib inline

In [ ]:
data['direction'] = np.sign(data['returns'])
#to_plot = ['midclose', 'returns', 'direction']
#data[to_plot].iloc[:100].plot(figsize=(10, 6),
#        subplots=True, style=['-', '-', 'ro'], title='EUR/USD');

In [ ]:
# the "patterns" = 2 ** lags
np.digitize(data[cols], bins=[0])[:10]

In [ ]:
2 ** len(cols)

In [ ]:
data.dropna(inplace=True)

In [ ]:
from sklearn import svm
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
#Added basic SVC model
#model = svm.SVC(C=100,random_state=111,


              # probability = True

#
#              )
#model = LogisticRegression(C=100)
#model = GaussianNB()
model = MLPClassifier(hidden_layer_sizes=[100,50 ,100], max_iter=200)

In [ ]:
data.info()

In [ ]:
#Fitting Model
%time model.fit(np.sign(data[cols]), np.sign(data['returns']))

In [ ]:
#Transforming model into 0 and 1
pred = model.predict(np.sign(data[cols]))
pred[:15]

In [ ]:
data['position'] = pred

In [ ]:
data['strategy'] = data['position'] * data['returns']
#data['strategy_tc'] = data['position'] * data['returns'] - data['spread']

In [ ]:
# unleveraged | no bid-ask spread or transaction costs | only in-sample
data[['returns', 'strategy' ]].cumsum().apply(np.exp).iplot()

In [ ]:
data['position'].value_counts()

# Adding a better way

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split


### Testing different techniques currently changed to half the standard deviation. Find a way to record and loop through different parameters

##### Current Change list
###### Changed from rolling mean to fixed mean (decrease in perfomance)
###### Changed from 1 standard deviation to half (decrease in perfomance)
###### Changed from 1 standard deviation to 2 (surprisingly robust out of sample perfomance) (Assuming the fact that because it normally just holds one position and doesnt change) ( Will need to test all these theories out)


In [ ]:
#mu = data['returns'].mean()
#v = data['returns'].std()

####Options for Bin vary for Volitility
##bins = [mu - v*2 , mu, mu + v*2 ]
#bins = [mu - v*0.5 , mu, mu + v*0.5 ]
#
#train_x, test_x, train_y, test_y = train_test_split(
#    data[cols].apply(lambda x: np.digitize(x, bins=bins)),
#    np.sign(data['returns']),
#    shuffle=True,
#    test_size=0.50, random_state=1111)

In [ ]:
mu = data['returns'].mean()
v = data['returns'].std()
bins = [mu - v, mu, mu + v]
#bins = [0]
train_x, test_x, train_y, test_y = train_test_split(
    data[cols].apply(lambda x: np.digitize(x, bins=bins)),
    np.sign(data['returns']),
    shuffle=True,
    test_size=0.50, random_state=1111)


In [ ]:
#data['returns'].tail()

In [ ]:
#v

In [ ]:
#bins

In [ ]:
#mu = data['returns'].mean()
#v = data['returns'].std()
#bins = [mu - 0.5 * v, mu, mu + 0.5 * v]
#train_x, test_x, train_y, test_y = train_test_split(
#    data[cols].apply(lambda x: np.digitize(x, bins=[0])),
#    np.sign(data['returns']),
#    shuffle=True,
#    test_size=0.50, random_state=111)

In [ ]:
#nos = 15
#mu = data['returns'].rolling(nos).mean()
#v = data['returns'].rolling(nos).std()
#def digits(s):
#    r = np.where(s < mu - v, 0, np.nan)
#    r = np.where(s > mu - v, 1, r)
#    r = np.where(s > mu, 2, r)
#    r = np.where(s > mu + v, 3, r)
#    return r

In [ ]:
#nos=2
#mu = data['returns'].rolling(nos).mean()
#v = data['returns'].rolling(nos).std()
#def digits(s):
#    r = np.where(s < mu - v, 0, np.nan)
#    r = np.where(s > mu - v, 1, r)
#    r = np.where(s > mu, 2, r)
#    r = np.where(s > mu + v, 3, r)
#    return r

In [ ]:
#data[cols].apply(digits)
#bins = [mu - v, mu, mu + v]
#train_x, test_x, train_y, test_y = train_test_split(
#    data[cols].apply(digits).dropna(),
#    np.sign(data['returns']).iloc[nos-1:],
#    test_size=0.50, random_state=111)

In [ ]:
train_x.sort_index(inplace=True)
train_y.sort_index(inplace=True)
test_x.sort_index(inplace=True)
test_y.sort_index(inplace=True)

In [ ]:
# the patterns = buckets ** lags
#train_x.head(5)

In [ ]:
#test_x.tail(5)

In [ ]:
4 ** len(cols)

In [ ]:
model.fit(train_x, train_y)

In [ ]:
train_pred = model.predict(train_x)

In [ ]:
accuracy_score(train_y, train_pred)

In [ ]:
test_pred = model.predict(test_x)

In [ ]:
accuracy_score(test_y, test_pred)

In [ ]:
pred = model.predict(data[cols].apply(
                    lambda x: np.digitize(x, bins=bins)).dropna())
#pred[:15]

In [ ]:
#data['position'] = 0.0

In [ ]:
pred = model.predict_proba(np.digitize(data[cols],bins=[0]))

In [ ]:
data.head()

In [ ]:
data.tail()

In [ ]:
data['position'] = pred*-1

In [ ]:
#data['position'] = 0.0
#data['position'].iloc[nos-1:] = pred

In [ ]:
data['strategy'] = data['position'] * data['returns']
data['creturns'] = data['returns'].cumsum().apply(np.exp)
data['cstrategy'] = data['strategy'].cumsum().apply(np.exp)
#absolute performance of the strategy
aperf = data['cstrategy'].iloc[-1]
# out-/underperformance of strategy
operf = aperf - data['creturns'].iloc[-1]


In [ ]:
#Backtester is incorrect it assumes transaction cost whether it held long or short but its a rough start but good enough
#data['after_commmision'] = ((data['position'] * data['returns'])-0.0002)

#data['after_commmision'] = data['position'] * data['returns']-np.abs(data['position'].diff()/2)*0.0001
#tc = 0.00001

#divide by two due to avoid double charge
spread = data['spread']/2
data['strategy_tc']=data['strategy']
trades = data['position'].diff() != 0.0
data['strategy_tc'] -= trades * spread

In [ ]:
# in-sample | unleveraged | no bid-ask spread or transaction costs
data.loc[train_x.index][['returns', 'strategy']].cumsum().apply(np.exp).iplot()



In [ ]:
# out-of-sample | unleveraged | no bid-ask spread or transaction costs
data.loc[test_x.index][['returns', 'strategy', ]].cumsum().apply(np.exp).iplot()

In [ ]:
data.loc[test_x.index][['returns', 'strategy','strategy_tc']].cumsum().apply(np.exp).iplot()

In [ ]:

#### OUT OF SAMPLE - OUT OF SAMPLE
data_oos.dropna(inplace=True)
pred_oos = model.predict(np.sign(data_oos[cols]))

data_oos['position'] = pred_oos

data_oos['strategy'] = data_oos['position'] * data_oos['returns']

data_oos_spread = (data_oos['spread']/2)
#tc = 0.00001
data_oos['strategy_tc']=data_oos['strategy']
trades_oos = data_oos['position'].diff() != 0.0
data_oos['strategy_tc'] -= trades_oos * data_oos_spread







data_oos[['returns', 'strategy','strategy_tc']].cumsum().apply(np.exp).iplot()

In [ ]:
# number of trades
sum(data['position'].diff() != 0)

In [ ]:
#Return Data Analysis

data.loc[test_x.index][['strategy_tc', 'strategy','returns']].apply(np.exp).iplot(kind='histogram',
                                                                   #subplots=True,
                                                                 barmode='overlay',
                                                                  #  shape=(3, 1),
                                                                   bins=math.sqrt(len(data['position'])),
                                                                       # bins= 600,
                                                                    opacity=0.5,
                                                                   histnorm='probability'
                                                                   )

### Adding Probability Classes

In [ ]:
#model = svm.SVC(C=100,random_state=111,
#
#
#              probability = True
#
#
#               )

model = MLPClassifier(hidden_layer_sizes=[200,25,25 ,200], max_iter=400)

#model = LogisticRegression(C=10000,
 #                          random_state = 111


#                          )

In [ ]:
model.fit(train_x, train_y)

In [ ]:
train_pred = model.predict(train_x)

In [ ]:
accuracy_score(train_y, train_pred)

In [ ]:
test_pred = model.predict(test_x)

In [ ]:
accuracy_score(test_y, test_pred)

In [ ]:
model.classes_

In [ ]:
pred_proba = model.predict_proba(np.digitize(data[cols],bins=[0]))
pred_proba[:8]

In [ ]:
probabilities = pd.DataFrame(pred_proba, columns =list(model.classes_))

In [ ]:
probabilities.iplot(kind='histogram',subplots=True)

In [ ]:

from sklearn.model_selection import cross_val_score

In [ ]:
pred_proba.max(axis=0)

In [ ]:
pred_proba.mean(axis=0)

In [ ]:
t = 0.52

pred = np.where(( pred_proba[:,0] > t) & (pred_proba[:,1]< 0.065 ), -1 ,0)

pred = np.where(( pred_proba[:,1] < 0.065 ) & (pred_proba[:, 2 ]> t ), 1 ,pred)

pred[:20]


In [ ]:
data['position'] = pred

In [ ]:
data['strategy'] = data['position'] * data['returns']

In [ ]:
#Accuracy Including Transaction
accuracy_score(data['position'], np.sign(data['strategy_tc']))

In [ ]:
#Accuracy Including Transaction
accuracy_score(data['position'], np.sign(data['strategy']))

In [ ]:
#Backtester is incorrect it assumes transaction cost whether it held long or short but its a rough start but good enough
#data['after_commmision'] = ((data['position'] * data['returns'])-0.0002)

#data['after_commmision'] = data['position'] * data['returns']-np.abs(data['position'].diff()/2)*0.0001
#tc = 0.00001
data['strategy_tc']=data['strategy']
spread = data['spread']/2
trades = data['position'].diff() != 0.0
data['strategy_tc'] -= trades * spread



In [ ]:
# in-sample | unleveraged | no bid-ask spread or transaction costs
data.loc[train_x.index][['returns', 'strategy']].cumsum().apply(np.exp).iplot()

In [ ]:
# out-of-sample | unleveraged | no bid-ask spread or transaction costs
data.loc[test_y.index][['returns', 'strategy']].cumsum().apply(np.exp).iplot()

In [ ]:
data.loc[test_x.index][['returns', 'strategy','strategy_tc']].cumsum().apply(np.exp).iplot()

In [ ]:

#### OUT OF SAMPLE - OUT OF SAMPLE
data_oos.dropna(inplace=True)
pred_proba_oos = model.predict_proba(np.sign(data_oos[cols]))

t = 0.50

pred_oos = np.where(( pred_proba_oos[:,0] > t) & (pred_proba_oos[:,1]< 0.065 ), -1 ,0)

pred_oos = np.where(( pred_proba_oos[:,1] < 0.065 ) & (pred_proba_oos[:, 2 ]> t ), 1 ,pred_oos)

pred[:20]

data_oos['position'] = pred_oos

data_oos['strategy'] = data_oos['position'] * data_oos['returns']

data_oos[['returns', 'strategy']].cumsum().apply(np.exp).iplot()


In [ ]:
#Return Data Analysis

data.loc[test_x.index][['strategy_tc', 'strategy','returns']].apply(np.exp).iplot(kind='histogram',
                                                                   #subplots=True,
                                                                 barmode='overlay',
                                                                  #  shape=(3, 1),
                                                                   bins=math.sqrt(len(data['position'])),
                                                                       # bins= 600,
                                                                    opacity=0.5,
                                                                   histnorm='probability'
                                                                   )

In [ ]:
#data = pd.DataFrame(candles[['askclose', 'bidclose']]).iplot()
data.head()

# AUTOMATED TRADING


### Streaming Data

In [ ]:
lags = 3


In [ ]:
def generate_features(df,lags):
    df["Returns"] = np.log(df["Mid"]/df["Mid"].shift(1))
    cols = []
    for lag in range(1, lags + 1):
        col = 'lag_%s' % lag
        df[col] = np.sign(df["Returns"].shift(lag))
        cols.append(col)
    df.dropna(inplace=True)
    return df , cols

In [ ]:
#1000 5 minute bars
candles = api.get_candles('EUR/USD', period = "m5", number = 1000)

In [ ]:
data = pd.DataFrame(candles[['askclose','bidclose']].mean(axis=1),columns=['Mid'])

In [ ]:
print(data)

In [ ]:
data , cols = generate_features(data ,lags)

In [ ]:
data.head()

In [ ]:
model.fit(data[cols],np.sign(data['Returns']))

In [ ]:
model.predict(data[cols])[:10]

# The basic idea

In [ ]:
data[cols].iloc[-1].values

In [ ]:
model.predict(data[cols].iloc[-1].values.reshape(1,-1))

# Opening another connection

In [ ]:
con2 = fxcmpy.fxcmpy(config_file='fxcm.cfg')

In [ ]:
to_show = ['tradeId','amountK','currency','grossPL','isBuy']

In [ ]:
#Initalising Variables
ticks = 0
position = 0
tick_data = pd.DataFrame()
tick_resam = pd.DataFrame()
#Changable Variables
unit_size= 2.5



### Added a better Autotrader (Has not tested live yet)

In [ ]:
#position = 0
#trades = 0
#ticks = 0
#min_length = lags + 1
#def auto_trader(data, dataframe):
#    global position, trades, ticks, min_length
#    ticks += 1
#    print(ticks, end=' ')
#    resam = dataframe.resample('10s', label='right').last().ffill()
#
#    if len(resam) > min_length:
#        min_length += 1
#        resam['mid'] = (resam['Bid'] + resam['Ask']) / 2
#        resam['returns'] = np.log(resam['mid'] / resam['mid'].shift(1))
#        features = np.sign(resam['returns'].iloc[-(lags+1):-1])
#        features = features.values.reshape(1, -1)
#        signal = model.predict(features)
#        print('\nNEW SIGNAL: {}'.format(signal))
#
#        if position in [0, -1]:
#            if signal == 1:
#                if position == -1:
#                    api.close_all_for_symbol(symbol)
#                api.create_market_buy_order(symbol, 1)
#                position = 1
#                print('{} | ***PLACING BUY ORDER***'.format(dt.datetime.now()))
#
#        elif position in [0, 1]:
#            if signal == -1:
#                if position == 1:
#                    api.close_all_for_symbol(symbol)
#                api.create_market_sell_order(symbol, 1)
#                position = -1
#                print('{} | ***PLACING SELL ORDER***'.format(dt.datetime.now()))
#
#        if ticks > 100:
#            pass

In [ ]:
#con2.subscribe_market_data('EUR/USD',(auto_trader,))

In [ ]:
def automated_trading(data,df):
    global lags, model, ticks
    global tick_data , tick_resam , to_show
    global position
    ticks +=1
    t = datetime.datetime.now()
    if ticks == 1:
        print('starting now at %s' %(str(t.time()) ))
    if ticks % 1000 == 0:
        print('%3d | %s | %7.5f | %7.5f' % (ticks, str(t.time()) , data['Rates'][0],data['Rates'][1]))


    #COLLECTING TICK DATA
    tick_data = tick_data.append(pd.DataFrame(
        {'Bid':data['Rates'][0],'Ask': data['Rates'][1],
        'High': data['Rates'][2],'Low': data['Rates'][3]},index=[t]))


    #Resample Tick Data
    tick_resam = tick_data[['Bid', 'Ask']].resample('5Min', label='right').last().ffill()
    tick_resam['Mid'] = tick_resam.mean(axis=1)

    if len(tick_resam) > lags + 2:
        #Generating Signal
    #print("Starting Signal Generation")

        tick_resam, cols = generate_features(tick_resam, lags)
        tick_resam['Prediction'] = model.predict(tick_resam[cols])
        #Generating a long position

        if tick_resam['Prediction'].iloc[-2] >= 0 and position == 0:

            print('Going Long for the first time')
            position = 1
            order = con2.create_market_buy_order('EUR/USD', unit_size)
            #trade = True
            #time.sleep(60)

        elif tick_resam['Prediction'].iloc[-2] >= 0 and position == -1:
            #api.close_all_for_symbol('EUR/USD')
            print('Going Long ')
            position = 1

            tradeId = con2.get_open_trade_ids()[0]
            pos = con2.get_open_position(tradeId)
            pos.close()
            order = con2.create_market_buy_order('EUR/USD', unit_size)
            #trade = True
            #time.sleep(60)
        #Entering a short position
        elif tick_resam['Prediction'].iloc[-2] <= 0 and position == 0:
            print('Going Short for the first time ')
            position = -1
            order = con2.create_market_sell_order('EUR/USD', unit_size)
            #trade = True
            #time.sleep(60)
        elif tick_resam['Prediction'].iloc[-2] <= 0 and position == 1:
            print('Going Short ')
            #api.close_all_for_symbol('EUR/USD')

            tradeId = con2.get_open_trade_ids()[0]
            pos = con2.get_open_position(tradeId)
            pos.close()
            position = -1
            order = con2.create_market_sell_order('EUR/USD', unit_size)
            #trade = True
            #time.sleep(60)
            #can Implement timer to shutout requests
            #so fair dont





#    if ticks > 39:
#        con2.unsubscribe_market_data('EUR/USD')
#        print("Closing all positions")
#        try:
#            con2.close_all()
#        except:
#            pass



In [ ]:
con2.subscribe_market_data('EUR/USD',(automated_trading,))

In [ ]:
tick_data.tail()



In [ ]:
tick_data.info()

In [ ]:
tick_resam.tail()


In [ ]:
tick_resam.info()

In [ ]:
try:
    print(con2.get_open_positions()[to_show])
except:
    print('no open positions')

In [ ]:
try:
    print(con2.get_closed_positions([to_show]))
except:
    print('no open positions')

In [ ]:
con2.unsubscribe_market_data('EUR/USD')

In [ ]:
len(tick_resam)